# Модуль 6.7 — Стоимость, латентность, маршрутизация моделей

Домашка к [лекции 6.7](https://itrubnikov.github.io/Train_of_Thought/docs/modules/06-7-cost-latency/). Вы соберёте **калькулятор стоимости вызова из `response.usage`** + **замер латентности (TTFT и tokens/sec)** + **роутер small→big с escalate-on-uncertainty и fallback**. Все блоки рабочие — `Run all` проходит целиком. Нужен API-ключ Anthropic — см. README.

Главный принцип: **числа берём из реального прогона** (usage-токены, замеры времени), а не из головы. Цены $/Mtok вынесены в один словарь `PRICES` — значения из лекции, но сверяйтесь с актуальным [прайсингом Anthropic](https://platform.claude.com/docs/en/about-claude/pricing).

## 0. Установка и клиент

Запустите ячейку. В Colab ключ берётся из Secrets (значок ключа слева, имя `ANTHROPIC_API_KEY`); локально — из `.env` (скопируйте `.env.example`).

In [ ]:
!pip -q install "anthropic>=0.69" openai python-dotenv pydantic
import os

# ключ: Colab Secrets -> переменная окружения; иначе .env
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Нет ANTHROPIC_API_KEY. Впишите его в .env (см. .env.example) или в Colab Secrets.")

from anthropic import Anthropic
client = Anthropic()                 # читает ANTHROPIC_API_KEY из окружения
MODEL = "claude-haiku-4-5"           # дешёвый для учёбы; флагман — "claude-opus-4-8"
FLAGSHIP = "claude-opus-4-8"
print("Готово, клиент создан.")

## Блок 1. Цены и калькулятор стоимости из usage

Счёт за вызов — это сумма трёх потоков токенов: `input` (то, что прислали) + `cache_read` (повторный префикс, ~0.1× цены input) + `output` (генерация, в ~5× дороже input). Цены $/Mtok берём из словаря `PRICES` (значения из лекции), а **токены — из реального `resp.usage`**.

`output` дороже, потому что генерация каждого токена прогоняет всю модель, а вход только читается. Поэтому короткий ответ дешевле длинного при том же входе.

In [ ]:
# Цены $/Mtok. Источник: claude-api skill bundle (first-party), сверяйтесь с прайсингом Anthropic.
# in = input, out = output. cache read ~0.1x от in, cache write ~1.25x (TTL 5 мин) / 2x (TTL 1 час).
PRICES = {
    "claude-haiku-4-5": {"in": 1,  "out": 5},    # дешёвый/быстрый
    "claude-opus-4-8":  {"in": 5,  "out": 25},   # флагман
    # "claude-sonnet-4-6": {"in": 3, "out": 15},  # баланс — можно добавить в задачах
}
CACHE_READ_MULT = 0.1    # чтение кэша ~0.1x цены input
CACHE_WRITE_MULT = 1.25  # запись кэша 1.25x (TTL 5 мин); 2x для TTL 1 час

def usage_to_dict(usage):
    """Достаёт 4 поля из resp.usage в обычный dict (поля cache_* могут быть None)."""
    return {
        "input_tokens": getattr(usage, "input_tokens", 0) or 0,
        "cache_read_input_tokens": getattr(usage, "cache_read_input_tokens", 0) or 0,
        "cache_creation_input_tokens": getattr(usage, "cache_creation_input_tokens", 0) or 0,
        "output_tokens": getattr(usage, "output_tokens", 0) or 0,
    }

def cost(usage, model):
    """Реальная цена вызова в долларах по usage-токенам и ценам PRICES.
    Принимает либо resp.usage, либо обычный dict с теми же ключами."""
    u = usage if isinstance(usage, dict) else usage_to_dict(usage)
    p = PRICES[model]
    dollars = (
        u["input_tokens"]               * p["in"]
        + u["cache_read_input_tokens"]     * p["in"] * CACHE_READ_MULT
        + u["cache_creation_input_tokens"] * p["in"] * CACHE_WRITE_MULT
        + u["output_tokens"]               * p["out"]
    ) / 1_000_000
    return dollars

print("Калькулятор готов. PRICES:", PRICES)

### Self-check: формула стоимости (работает БЕЗ ключа)

Прежде чем доверять калькулятору на реальных вызовах, проверим формулу `assert`-ом на известных числах токенов. Если ячейка упала — в формуле ошибка, чинить до прогона API.

In [ ]:
# Чистая арифметика, ключ не нужен. Считаем вручную и сверяем с cost().

# 1) Haiku, 1000 input + 200 output: 1000*1 + 200*5 = 2000 микро-$ = $0.002
u1 = {"input_tokens": 1000, "cache_read_input_tokens": 0,
      "cache_creation_input_tokens": 0, "output_tokens": 200}
assert abs(cost(u1, "claude-haiku-4-5") - 0.002) < 1e-12, cost(u1, "claude-haiku-4-5")

# 2) Opus, 500 input + 10 output: 500*5 + 10*25 = 2750 микро-$ = $0.00275
#    output ровно в 5x дороже input на Opus — видно на этом примере
u2 = {"input_tokens": 500, "cache_read_input_tokens": 0,
      "cache_creation_input_tokens": 0, "output_tokens": 10}
assert abs(cost(u2, "claude-opus-4-8") - 0.00275) < 1e-12, cost(u2, "claude-opus-4-8")

# 3) Кэш: 100 input + 400 cache_read + 10 output на Haiku
#    100*1 + 400*1*0.1 + 10*5 = 100 + 40 + 50 = 190 микро-$ = $0.00019
u3 = {"input_tokens": 100, "cache_read_input_tokens": 400,
      "cache_creation_input_tokens": 0, "output_tokens": 10}
assert abs(cost(u3, "claude-haiku-4-5") - 0.00019) < 1e-12, cost(u3, "claude-haiku-4-5")

# 4) Один и тот же ответ на Opus стоит в 5x дороже, чем на Haiku (одинаковые токены)
same = {"input_tokens": 500, "cache_read_input_tokens": 0,
        "cache_creation_input_tokens": 0, "output_tokens": 10}
ratio = cost(same, "claude-opus-4-8") / cost(same, "claude-haiku-4-5")
assert abs(ratio - 5.0) < 1e-9, ratio

print("Self-check пройден: формула стоимости верна (без ключа).")
print(f"  Opus дороже Haiku ровно в {ratio:.1f}x при одинаковых токенах.")

### Классификатор писем — один вызов, печатаем usage и цену

Канонический классификатор писем из модуля 6: категория письма одним словом. `max_tokens=10` — нам нужен один ярлык, не рассуждение. Печатаем реальные токены из `resp.usage` и считаем цену через `cost()`.

In [ ]:
CLASSIFY_SYSTEM = (
    "Ты классифицируешь входящие письма по категории. "
    "Ответь РОВНО одним словом из набора: рабочее, личное, спам, уведомление, жалоба. "
    "Без пояснений и знаков препинания."
)

def classify(text, model=MODEL, max_tokens=10):
    """Классифицирует письмо. Возвращает (ярлык, resp). Токены — в resp.usage."""
    resp = client.messages.create(
        model=model, max_tokens=max_tokens,
        system=CLASSIFY_SYSTEM,
        messages=[{"role": "user", "content": f"Письмо: {text}"}],
    )
    label = resp.content[0].text.strip()
    return label, resp

EMAIL = "Здравствуйте! Напоминаю про дедлайн по отчёту в пятницу, нужна ваша часть."
label, resp = classify(EMAIL)
u = usage_to_dict(resp.usage)
print(f"Ярлык: {label}")
print(f"usage: input={u['input_tokens']}  output={u['output_tokens']}  "
      f"cache_read={u['cache_read_input_tokens']}")
print(f"Цена этого вызова на {MODEL}: ${cost(resp.usage, MODEL):.6f}")

## Блок 2. Один классификатор, два счёта (Haiku vs Opus)

Тот же промпт и набор писем гоняем на дешёвой (Haiku) и дорогой (Opus) модели. Печатаем ярлык и цену для каждой, в конце — суммарную цену и где ярлыки разошлись.

Заранее знать точные доллары нельзя — их печатает прогон. Ожидаемая суть: на простых письмах ярлыки совпадут, но Opus стоит впятеро дороже за тот же ответ. Разница в качестве проявляется на трудных, неоднозначных письмах — это и есть повод для маршрутизации (блок 6).

In [ ]:
EMAILS = [
    "Напоминаю про дедлайн по отчёту в пятницу.",              # простое: рабочее
    "Привет! Как дела, давно не виделись, пойдём в кино?",      # простое: личное
    "ВЫ ВЫИГРАЛИ МИЛЛИОН! Срочно перейдите по ссылке прямо сейчас!",  # простое: спам
    "Ваш заказ #4471 отправлен и будет доставлен завтра.",      # простое: уведомление
    # трудные/неоднозначные — тут модели могут разойтись:
    "Спасибо за 'отличный' сервис, ждал доставку три недели, очень доволен.",  # сарказм -> жалоба
    "По поводу нашего вчерашнего разговора: высылаю файл, посмотри на досуге.",  # рабочее/личное на грани
]

totals = {MODEL: 0.0, FLAGSHIP: 0.0}
rows = []
for text in EMAILS:
    label_h, resp_h = classify(text, model=MODEL)
    label_o, resp_o = classify(text, model=FLAGSHIP)
    c_h, c_o = cost(resp_h.usage, MODEL), cost(resp_o.usage, FLAGSHIP)
    totals[MODEL]    += c_h
    totals[FLAGSHIP] += c_o
    same = "=" if label_h == label_o else "РАЗОШЛИСЬ"
    rows.append((text[:42], label_h, label_o, c_h, c_o, same))

print(f"{'письмо':<44}{'Haiku':<14}{'Opus':<14}{'совпало?'}")
for text, lh, lo, ch, co, same in rows:
    print(f"{text:<44}{lh:<14}{lo:<14}{same}")

print("\n--- суммарная цена за весь набор ---")
print(f"Haiku:  ${totals[MODEL]:.6f}")
print(f"Opus:   ${totals[FLAGSHIP]:.6f}")
if totals[MODEL] > 0:
    print(f"Opus дороже Haiku в {totals[FLAGSHIP]/totals[MODEL]:.1f}x на этом наборе.")
n_diff = sum(1 for r in rows if r[5] != '=')
print(f"Ярлыки разошлись на {n_diff} из {len(rows)} писем — кандидаты на эскалацию.")

## Блок 3. Оценка входа ДО запуска: count_tokens

Прежде чем платить за тысячу вызовов, оцените вход. `client.messages.count_tokens(...)` на **той же модели** возвращает `input_tokens` — умножаете на цену input и видите потолок входной стоимости.

**Не tiktoken.** Это чужой (OpenAI) токенайзер; на Claude он занижает количество токенов на ~15–20% (на коде и не-английском больше), потому что токенайзер модель-специфичен. `output` заранее точно не посчитать — оцениваете сверху через `max_tokens` и меряете по факту из `resp.usage.output_tokens`.

In [ ]:
# Оценим вход для одного письма ДО вызова
est = client.messages.count_tokens(
    model=MODEL,
    system=CLASSIFY_SYSTEM,
    messages=[{"role": "user", "content": f"Письмо: {EMAIL}"}],
)
in_price = PRICES[MODEL]["in"]
est_cost = est.input_tokens * in_price / 1_000_000
print(f"count_tokens на {MODEL}: input_tokens={est.input_tokens}")
print(f"Оценка входной стоимости: ${est_cost:.6f}  (только вход, output меряем по факту)")

# Прикинем потолок входа на 1000 таких писем
print(f"На 1000 писем вход обойдётся ~${est_cost * 1000:.4f} (output добавится сверху по resp.usage).")
print("\nНапоминание: output заранее не посчитать — это генерация. Меряем фактический output_tokens из resp.usage.")

## Блок 4. Латентность: TTFT и throughput через стриминг

Латентность — это два разных числа. **TTFT** (time to first token) — сколько ждать первого токена; **throughput** (tokens/sec) — как быстро печатается остальное. Пользователь чувствует в первую очередь TTFT, и стриминг его снижает.

Стриминг ещё и спасает от HTTP-таймаута SDK на длинном выводе: вывод > 16K токенов нужно стримить обязательно (Opus до 128K только в стриме). Замеряем через `time.perf_counter()`: TTFT — до первого text-чанка, throughput = output_tokens / общее время. Числа — из реального прогона.

In [ ]:
from time import perf_counter

def measure_latency(prompt, model=MODEL, max_tokens=300):
    """Стримит ответ, замеряет TTFT и throughput. Числа из реального прогона."""
    t0 = perf_counter()
    ttft = None
    with client.messages.stream(
        model=model, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        for _text in stream.text_stream:
            if ttft is None:
                ttft = perf_counter() - t0      # первый text-чанк пришёл
        final = stream.get_final_message()
    total = perf_counter() - t0
    out_tokens = final.usage.output_tokens
    tps = out_tokens / total if total > 0 else 0.0
    return {"ttft_s": ttft, "total_s": total, "output_tokens": out_tokens, "tokens_per_sec": tps}

# Берём задачу с заметным выводом, чтобы throughput был осмысленным
m = measure_latency("Объясни в 5-6 предложениях, чем стриминг ответа лучше для UX.", model=MODEL)
print(f"Модель: {MODEL}")
print(f"  TTFT (до первого токена): {m['ttft_s']:.3f} c")
print(f"  Всего:                    {m['total_s']:.3f} c")
print(f"  output_tokens:            {m['output_tokens']}")
print(f"  throughput:               {m['tokens_per_sec']:.1f} tokens/sec")
print("\nTTFT лечится стримингом; throughput — это скорость печати. Это РАЗНЫЕ рычаги.")

## Блок 5. Prompt caching как рычаг стоимости

Стабильный system-префикс (инструкция классификатора, few-shot) можно закэшировать: первый запрос его **пишет** (`cache_creation_input_tokens`, ~1.25×), второй — **читает** почти бесплатно (`cache_read_input_tokens`, ~0.1×). Со 2-го запроса при TTL 5 мин кэш окупается.

Минимальную длину кэшируемого префикса в число не зашиваем — она зависит от модели (порядка нескольких тысяч токенов). Поэтому ниже мы раздуваем `system` с запасом и **проверяем эмпирически** по `cache_read_input_tokens`. Тихие инвалидаторы (`datetime.now()`, uuid, несортированный json) ломают кэш молча — потому и проверяем по факту, а не верим на слово.

In [ ]:
# Раздуваем стабильный префикс заведомо выше минимума кэширования
BIG_GUIDELINE = (CLASSIFY_SYSTEM + "\n"
    "Правила: рабочее — про задачи/проекты/дедлайны; личное — от друзей/семьи; "
    "спам — навязчивая реклама/мошенничество; уведомление — автоматическое от сервиса; "
    "жалоба — недовольство, в т.ч. с сарказмом. ") * 80

def classify_cached(text, model=MODEL):
    return client.messages.create(
        model=model, max_tokens=10,
        system=[{"type": "text", "text": BIG_GUIDELINE,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Письмо: {text}"}],
    )

r1 = classify_cached("Напоминаю про дедлайн в пятницу.")
u1 = usage_to_dict(r1.usage)
print("1-й запрос (пишет кэш):")
print(f"  cache_creation={u1['cache_creation_input_tokens']}  "
      f"cache_read={u1['cache_read_input_tokens']}  input={u1['input_tokens']}")
print(f"  цена: ${cost(r1.usage, MODEL):.6f}")

r2 = classify_cached("Как дела, пойдём в кино?")   # тот же префикс -> должен читаться из кэша
u2 = usage_to_dict(r2.usage)
print("\n2-й запрос (должен читать кэш):")
print(f"  cache_creation={u2['cache_creation_input_tokens']}  "
      f"cache_read={u2['cache_read_input_tokens']}  input={u2['input_tokens']}  <- read > 0 = кэш сработал")
print(f"  цена: ${cost(r2.usage, MODEL):.6f}")

if u2['cache_read_input_tokens'] > 0:
    saved = (u2['cache_read_input_tokens'] * PRICES[MODEL]['in'] * (1 - CACHE_READ_MULT)) / 1_000_000
    print(f"\nКэш сработал. На 2-м запросе сэкономлено ~${saved:.6f} против полной цены за этот префикс.")
else:
    print("\ncache_read=0: префикс короче минимума ИЛИ сработал тихий инвалидатор. Проверьте байты префикса.")

## Блок 6. Роутер small→big с escalate-on-uncertainty

Главная идея экономии — не брать флагман на каждый вход. Дешёвая модель (Haiku) обрабатывает поток и возвращает **не только ярлык, но и уверенность**; неуверенные/трудные случаи эскалируются на дорогую (Opus). Качество там, где нужно; цена — везде, где не нужно.

Уверенность получаем через structured output (Pydantic-схема из 6.5): `label` + `confidence` (0..1). Порог `THRESHOLD` решает, эскалировать ли. В конце сравниваем цену роутера с «всё на Opus».

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Verdict(BaseModel):
    label: Literal["рабочее", "личное", "спам", "уведомление", "жалоба"]
    confidence: float   # 0..1 — насколько модель уверена

THRESHOLD = 0.75
ROUTE_SYSTEM = (
    "Ты классифицируешь письмо по категории и честно оцениваешь свою уверенность 0..1. "
    "Если письмо неоднозначное (сарказм, смешанная тема) — ставь низкую уверенность."
)

def classify_with_conf(text, model):
    resp = client.messages.parse(
        model=model, max_tokens=128,
        system=ROUTE_SYSTEM,
        messages=[{"role": "user", "content": f"Письмо: {text}"}],
        output_format=Verdict,
    )
    return resp.parsed_output, resp

def route(text):
    """small->big: сначала Haiku; если не уверена — эскалация на Opus.
    Возвращает (label, путь, цена_роутера)."""
    v, resp_h = classify_with_conf(text, MODEL)
    spent = cost(resp_h.usage, MODEL)
    if v.confidence >= THRESHOLD:
        return v.label, f"Haiku (conf={v.confidence:.2f})", spent
    # низкая уверенность -> платим за дорогую модель
    v2, resp_o = classify_with_conf(text, FLAGSHIP)
    spent += cost(resp_o.usage, FLAGSHIP)
    return v2.label, f"Haiku->Opus (conf {v.confidence:.2f}->{v2.confidence:.2f})", spent

router_total = 0.0
opus_only_total = 0.0
escalated = 0
print(f"{'письмо':<44}{'ярлык':<13}{'маршрут'}")
for text in EMAILS:
    label, path, spent = route(text)
    router_total += spent
    if "Opus" in path:
        escalated += 1
    # для сравнения: сколько стоило бы то же письмо целиком на Opus
    _, resp_o = classify_with_conf(text, FLAGSHIP)
    opus_only_total += cost(resp_o.usage, FLAGSHIP)
    print(f"{text[:42]:<44}{label:<13}{path}")

print("\n--- экономия роутера ---")
print(f"Роутер small->big:   ${router_total:.6f}  (эскалаций: {escalated} из {len(EMAILS)})")
print(f"Всё на Opus:         ${opus_only_total:.6f}")
if opus_only_total > 0:
    save_pct = (1 - router_total / opus_only_total) * 100
    print(f"Экономия роутера:    {save_pct:.0f}% против «флагман на всё».")

## Блок 7 (опц.). Fallback на ошибки и Batch API — демо + честное описание

**Fallback.** SDK сам ретраит 408/409/429/5xx с экспоненциальным backoff (`max_retries=2` по умолчанию) — кастомный цикл нужен только за пределами этого. На `529 overloaded_error` разумен фолбэк на менее загруженную модель (Haiku часто свободнее), чем тупой повтор. Ниже — try/except с фолбэком модели.

**Batch API.** Для НЕ латентно-чувствительных задач (офлайн-разметка корпуса, ночная обработка) — скидка **−50% на все токены**. Лимиты: ≤100 000 запросов или 256 MB на батч; обычно <1 часа, максимум 24 часа; результаты хранятся 29 дней. Поддерживает всё (vision, tools, caching). Ниже — скелет batch-запроса без реального запуска большого батча (демо стоит денег и времени, а суть в форме вызова).

In [ ]:
import anthropic

def classify_with_fallback(text, primary=FLAGSHIP, fallback=MODEL):
    """Пытается на дорогой модели; при перегрузке/лимите — фолбэк на дешёвую/свободную.
    SDK уже ретраит сам; этот фолбэк — для случая, когда повторы не помогли."""
    try:
        label, resp = classify(text, model=primary)
        return label, primary, cost(resp.usage, primary)
    except (anthropic.RateLimitError, anthropic.InternalServerError) as e:
        # 429 / 5xx / 529 -> идём на менее загруженную модель
        print(f"  {type(e).__name__} на {primary} -> фолбэк на {fallback}")
        label, resp = classify(text, model=fallback)
        return label, fallback, cost(resp.usage, fallback)

label, used_model, c = classify_with_fallback("Ваш заказ отправлен.")
print(f"Сработал путь: {used_model}  ярлык={label}  цена=${c:.6f}")

# --- Batch API: СКЕЛЕТ (не запускаем большой батч ради денег/времени) ---
# from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
# from anthropic.types.messages.batch_create_params import Request
#
# requests = [
#     Request(
#         custom_id=f"email-{i}",
#         params=MessageCreateParamsNonStreaming(
#             model=MODEL, max_tokens=10,
#             system=CLASSIFY_SYSTEM,
#             messages=[{"role": "user", "content": f"Письмо: {text}"}],
#         ),
#     )
#     for i, text in enumerate(EMAILS)
# ]
# batch = client.messages.batches.create(requests=requests)
# # ... опрос client.messages.batches.retrieve(batch.id) до processing_status == "ended"
# # ... сбор client.messages.batches.results(batch.id) -> цена в 2 раза ниже онлайна
print("\nBatch API: −50% на все токены для офлайна. Скелет вызова — в комментариях выше.")

## Блок 8 (опц.). Дешёвый сторонний провайдер: MiniMax (OpenAI-совместимый)

Маршрутизация — это не только «Haiku → Opus внутри Anthropic». Иногда самый дешёвый жизнеспособный вариант для низкорискового потока — сторонний провайдер. **MiniMax** OpenAI-совместим: тот же код, что у `openai`, меняете три вещи — `base_url`, `model` и ключ. Цена `MiniMax-M2.5` — порядка **$0.15 / $0.90** за Mtok (вход/выход): это ~6× дешевле Haiku и ~30× дешевле Opus по входу.

Три честных нюанса:
- **`usage` называется иначе.** В OpenAI-формате это `prompt_tokens` / `completion_tokens`, а не `input_tokens` / `output_tokens` как у Anthropic. Не перепутайте при подсчёте цены.
- **Дёшево ≠ всегда правильно.** Сравните качество на своей задаче (трудные письма), латентность и требования к данным — ровно как мы делали Haiku vs Opus. Дешёвый провайдер хорош там, где качество не страдает.
- **Это и есть работа шлюза.** Переключение между провайдерами по правилу операционализирует модуль 6.3 (OpenRouter / шлюз); 6.7 отвечает «по какому правилу и ради какой экономии».

Блок опциональный: нужен отдельный `MINIMAX_API_KEY` (platform.minimax.io). Без него ячейка мягко пропускается — `Run all` не ломается.

In [ ]:
# MiniMax — OpenAI-совместимый провайдер: тот же код, меняем base_url + model + ключ.
# Опционально: нужен отдельный MINIMAX_API_KEY. Без него блок пропускается, Run all не ломается.
import os

MINIMAX_KEY = os.getenv("MINIMAX_API_KEY")
if not MINIMAX_KEY:
    print("MINIMAX_API_KEY не задан — блок 8 пропущен (он опциональный).")
    print("Ключ заводится на platform.minimax.io; впишите его в .env / Secrets, чтобы сравнить цену.")
else:
    from openai import OpenAI

    # Цена $/Mtok. Источник: openrouter.ai/minimax (2026-06). Цены двигаются — сверяйтесь с прайсингом MiniMax.
    MM_MODEL = "MiniMax-M2.5"                       # новейшая — "MiniMax-M3" (1M контекст); поменяйте строку
    MM_PRICE = {"MiniMax-M2.5": {"in": 0.15, "out": 0.90}}

    mm = OpenAI(base_url="https://api.minimax.io/v1", api_key=MINIMAX_KEY)

    r = mm.chat.completions.create(
        model=MM_MODEL, max_tokens=10,
        messages=[
            {"role": "system", "content": CLASSIFY_SYSTEM},      # тот же контракт, что у Anthropic-классификатора
            {"role": "user", "content": f"Письмо: {EMAILS[0]}"}, # то же письмо, что в блоке 2
        ],
    )

    # OpenAI-формат usage: prompt_tokens / completion_tokens (НЕ input/output как у Anthropic).
    u = r.usage
    p = MM_PRICE[MM_MODEL]
    mm_cost = (u.prompt_tokens * p["in"] + u.completion_tokens * p["out"]) / 1_000_000

    print(f"{MM_MODEL} -> {r.choices[0].message.content.strip()}")
    print(f"  prompt={u.prompt_tokens}  completion={u.completion_tokens}  цена=${mm_cost:.8f}")
    print(f"  цены $/Mtok для сравнения:  MiniMax-M2.5 {p['in']}/{p['out']}  |  Haiku 1/5  |  Opus 5/25")
    print("  Дёшево, но проверьте ярлык на трудных письмах из EMAILS — дешевизна оправдана там, где качество не страдает.")


## Задачи — доработайте рабочий код

Семь блоков работают. Теперь учимся, меняя готовое (сделайте минимум 4 из 6):

1. **Стоимость: длинный CoT.** В `classify` поднимите `max_tokens` до 400 и попросите модель рассуждать вслух (добавьте в system «сначала рассуждай по шагам»). Прогоните на одном письме на Haiku и на Opus, сравните цену с коротким вариантом. Убедитесь, что доля output резко выросла — output в ~5× дороже input.
2. **Оценка входа: count_tokens vs факт.** Сравните `count_tokens` (оценка входа) с фактическим `resp.usage.input_tokens` после реального вызова того же письма. Совпали? Запишите вывод, почему это важно для прогноза счёта.
3. **Латентность: эффект max_tokens.** Вызовите `measure_latency` с `max_tokens=50` и `max_tokens=600` на одном промпте. Как меняются TTFT и throughput? Запишите: TTFT почти не зависит от длины, а общее время — да.
4. **Кэш: сломайте его.** Допишите в начало `BIG_GUIDELINE` текущее время (`from datetime import datetime; str(datetime.now())`) и повторите блок 5. Убедитесь, что `cache_read_input_tokens` стал 0 (тихий инвалидатор) — и цена 2-го запроса выросла.
5. **Роутер: подвигайте порог.** Поставьте `THRESHOLD = 0.5`, затем `0.95`, прогоните блок 6. Как меняется число эскалаций и итоговая цена? Найдите порог, где экономия максимальна без явных ошибок ярлыка.
6. **Третья модель в роутере.** Добавьте `claude-sonnet-4-6` (`{"in":3,"out":15}`) в `PRICES` и сделайте трёхступенчатый роутер Haiku→Sonnet→Opus. Сравните цену с двухступенчатым. Где «средняя» модель окупается?

Каждая задача — правка рабочего кода. По каждой запишите короткий вывод.

7. **(опц., бонус) MiniMax как самый дешёвый tier.** Прогоните блок 8 (нужен `MINIMAX_API_KEY`) и встройте `MiniMax-M2.5` в роутер из блока 6 как нулевой tier: сначала MiniMax, на неуверенности — Haiku, затем Opus. Посчитайте суммарную цену трёхуровневого роутера против «всё на Opus».


## Что сдать

- [ ] Ноутбук прогнан целиком (`Run all`) — калькулятор, замер латентности и роутер отработали.
- [ ] Зафиксирован числовой результат прогона: реальная цена из `usage` (например, суммарная цена Haiku vs Opus из блока 2 и экономия роутера из блока 6).
- [ ] Self-check формулы стоимости прошёл (ячейка с `assert`-ами, работает без ключа).
- [ ] Сделаны задачи-доработки (мин. 4 из 6) с короткими выводами.
- [ ] Ключ нигде не захардкожен — только `.env` / Secrets.

Вывод одной фразой запишите в ячейку ниже: где разброс стоимости между «дёшево» и «дорого» оказался оправдан качеством, а где это были деньги на ветер.

_(Ваш вывод одной фразой здесь.)_